# UCI Credit Card Default – Feature Engineering for PD

**Purpose:** Build a PD (Probability of Default) dataset from the UCI "Default of Credit Card Clients" data so it can be trained with the same XGBoost pipeline as Lending Club (e.g. reuse logic from `02_pd_xgboost_training.ipynb` with a different data path and feature list).

**Steps:**
1. Load UCI data (CSV/XLS from `data/credit_risk_pd/UCI/` or download from UCI ML repo).
2. Define target: `default payment next month` (1 = default, 0 = non-default).
3. Select no-leakage features via `credit_risk.feature_engineering.uci_features.get_uci_feature_names()`.
4. Train/val/test split (random stratified); optional outlier caps.
5. Save `data/credit_risk_pd/UCI/processed/uci_credit_engineered.parquet` (columns: features + `default` + `split`).

**Downstream:** Use this parquet in a UCI-specific training notebook (same structure as 02: load parquet, use `get_uci_feature_names()`, train XGBoost, save model).

## 1. Setup and load raw data

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(".").resolve() if (Path(".").resolve() / "credit_risk").exists() else Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data" / "credit_risk_pd" / "UCI"
OUTPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Prefer CSV (header on row 2 in original XLS)
csv_path = DATA_DIR / "default_of_credit_card_clients.csv"
xls_path = DATA_DIR / "default_of_credit_card_clients.xls"

if csv_path.exists():
    df_raw = pd.read_csv(csv_path)
elif xls_path.exists():
    df_raw = pd.read_excel(xls_path, header=1)
else:
    raise FileNotFoundError(
        "Place UCI data (default_of_credit_card_clients.csv or .xls) under data/credit_risk_pd/UCI/. "
        "Download: https://archive.ics.uci.edu/ml/datasets/default+of+credit+card+clients"
    )

# Standard UCI column name for target (may have space)
target_col = "default payment next month"
if target_col not in df_raw.columns:
    cand = [c for c in df_raw.columns if "default" in c.lower()]
    target_col = cand[0] if cand else df_raw.columns[-1]
if target_col != "default":
    df_raw = df_raw.rename(columns={target_col: "default"})
# Drop ID if present (not a feature)
if "ID" in df_raw.columns:
    df_raw = df_raw.drop(columns=["ID"])
print(f"Loaded {len(df_raw):,} rows. Columns: {list(df_raw.columns)}")

## 2. Target and feature set

In [ ]:
from credit_risk.feature_engineering.uci_features import get_uci_feature_names

feature_names = get_uci_feature_names()
available = [c for c in feature_names if c in df_raw.columns]
missing = [c for c in feature_names if c not in df_raw.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. UCI data may use different names.")

df = df_raw.copy()
df["default"] = df["default"].astype(int).clip(0, 1)
print(df["default"].value_counts())
print(f"Features: {len(feature_names)}")

## 3. Coerce types and optional caps

In [ ]:
for c in feature_names:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Optional: cap extreme values (e.g. LIMIT_BAL, AGE)
df["LIMIT_BAL"] = df["LIMIT_BAL"].clip(lower=0, upper=df["LIMIT_BAL"].quantile(0.999))
df["AGE"] = df["AGE"].clip(lower=21, upper=79)
print("Numeric coercion and caps applied.")

## 4. Train / val / test split and fill NaN

In [ ]:
from sklearn.model_selection import train_test_split

X = df[feature_names].copy()
y = df["default"]
for c in X.columns:
    if X[c].isna().any():
        X[c] = X[c].fillna(X[c].median())

X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=42, stratify=y_rest)
print(f"Train {len(X_train):,} / Val {len(X_val):,} / Test {len(X_test):,}")

df_out = pd.concat([X_train, X_val, X_test], axis=0, ignore_index=True)
df_out["default"] = pd.concat([y_train, y_val, y_test], axis=0, ignore_index=True)
df_out["split"] = ["train"] * len(X_train) + ["valid"] * len(X_val) + ["test"] * len(X_test)

## 5. Save engineered parquet

In [ ]:
out_path = OUTPUT_DIR / "uci_credit_engineered.parquet"
df_out.to_parquet(out_path, index=False)
print(f"Saved to {out_path} ({len(df_out):,} rows, {len(feature_names)} features + default + split).")
print("To train XGBoost: load this parquet, use get_uci_feature_names(), then same flow as 02_pd_xgboost_training.ipynb with DATA_PATH and feature_names set for UCI.")